## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from calculations.config.model_config import (R, CONDITIONS, MICROSTRUCTURE,
                                              OXIDE_DEFECTS, build_simulation_config)
from calculations.oxide_permeation import (get_oxide_properties_at_T, get_metal_properties_at_T,
                                           molecular_diffusion_flux)
from calculations.permeation_calc import calculate_simple_metal_flux, calculate_defective_metal_flux
from calculations.interface_solver import calculate_oxide_metal_system
from calculations.parallel_oxide_defect_paths import (calculate_parallel_path_flux,
                                                      calculate_parallel_path_flux_defective_metal)
from calculations.wall_labels import label_model1, label_model2
from calculations.permeability import apparent_permeability, UNITS_SIEVERTS, UNITS_HENRY
import pandas as pd
from IPython.display import display
from calculations.figures import results_dir, save_figure
FIGS = results_dir("effect-of-temperature")                             # figures and fit tables go in this folder

SIM          = build_simulation_config()
P_up, P_down = CONDITIONS['P_upstream'], SIM['P_downstream']
L_ox, L_m    = SIM['L_oxide'], SIM['L_metal']
T = np.linspace(*CONDITIONS['T_range'], CONDITIONS['n_points'])
x = 1000 / T

LEVELS = {   # level: (title, wall thickness for Φ_app)
    'L1':  ('Level 1: Perfect metal (lattice diffusion)',          L_m),
    'L2a': ('Level 2a: Perfect oxide alone (molecular H$_2$)',     L_ox),
    'L2b': ('Level 2b: Perfect oxide + perfect metal',             L_ox + L_m),
    'L3':  ('Level 3: Defective oxide + perfect metal',            L_ox + L_m),
    'L4':  ('Level 4: Defective metal (traps + grain boundaries)', L_m),
    'L5':  ('Level 5: Defective oxide + defective metal',          L_ox + L_m),
}

In [ ]:
def solve(level, Tk):
    """Flux and regime label for one level at one temperature."""
    if level.endswith('L6'):
        return solve_m2(level, Tk)                       # Model 2 levels, defined further down
    ox = get_oxide_properties_at_T(SIM['oxide_name'], Tk); ox['thickness'] = L_ox
    me = get_metal_properties_at_T(SIM['metal_name'], Tk); me['thickness'] = L_m
    D, K  = me['D_metal'], me['K_s_metal']
    micro = dict(microstructure_params=MICROSTRUCTURE, lattice_density=MICROSTRUCTURE['N_L'], mode='both')
    if level == 'L1':  return calculate_simple_metal_flux(D, K, L_m, P_up, P_down)['flux'], label_model1('L1')
    if level == 'L2a': return molecular_diffusion_flux(ox['D_ox'], ox['K_ox'], L_ox, P_up, P_down), label_model1('L2a')
    if level == 'L2b': r = calculate_oxide_metal_system(P_up, P_down, ox, me)
    if level == 'L3':  r = calculate_parallel_path_flux(P_up, P_down, ox, me, OXIDE_DEFECTS)
    if level == 'L4':  r = calculate_defective_metal_flux(D_lattice=D, K_s=K, metal_thickness=L_m,
                                                          P_up=P_up, P_down=P_down, temperature=Tk, **micro)
    if level == 'L5':  r = calculate_parallel_path_flux_defective_metal(P_up, P_down, ox, me,
                                                                        OXIDE_DEFECTS, Tk, **micro)
    return r.get('flux_total', r.get('flux')), label_model1(level, r, P_up, P_down, ox, me, OXIDE_DEFECTS, Tk, micro)

def sweep(level):
    J, regime = map(np.array, zip(*[solve(level, Tk) for Tk in T]))
    if level == 'L2a':                                   # Henry: J·L/ΔP
        return J, J * L_ox / (P_up - P_down), UNITS_HENRY, regime
    return J, apparent_permeability(J, P_up, P_down, LEVELS[level][1]), UNITS_SIEVERTS, regime

def arrhenius(x, y):
    fit = stats.linregress(x, np.log(y))
    return -fit.slope * R * 1000, np.exp(fit.intercept), fit.rvalue**2   # Ea (J/mol), A, R²


## 2: Physics (one solver call per level, then the sweep and the fit)

In [ ]:
mpl.rcParams.update({
    'font.size': 14, 'font.weight': 'bold', 'mathtext.default': 'bf',
    'axes.labelweight': 'bold', 'axes.titleweight': 'bold', 'axes.linewidth': 1.8,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'ytick.right': True,
    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,
    'legend.frameon': False, 'legend.fontsize': 11, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
INK, GREY, SHADE = '#0b0b0b', '#52514e', '#e1e0d9'

# Regime labels name the dominant path and the limiting step inside it (calculations/wall_labels.py):
# colour = limiting step, hollow = a defect path carries most of the flux, diamond = trap-controlled.
from calculations.wall_labels import describe as look, fitted
UNITS_TEX = {UNITS_SIEVERTS: r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1/2}$',
             UNITS_HENRY:    r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1}$'}


## 3: Style

In [ ]:
from matplotlib.lines import Line2D

LINES, ROLES = ['-', '--', ':'], ['low', 'mid', 'high']   # fit line style per condition, in order

def p_tag(Pc):
    """Short pressure label: 1e-07 -> '1e-7 Pa', 1.0 -> '1e0 Pa', 1e+07 -> '1e7 Pa'."""
    return f'{Pc:.0e}'.replace('e+0', 'e').replace('e-0', 'e-') + ' Pa'

def framed(nrows, figsize, sharex=True):
    """Figure with two legend strips under the title and an nrows x 2 grid of panels below them."""
    fig  = plt.figure(figsize=figsize, layout='constrained')
    gs   = fig.add_gridspec(nrows + 2, 2, height_ratios=[0.001, 0.001] + [1] * nrows)
    keys = [fig.add_subplot(gs[k, :]) for k in (0, 1)]
    for k in keys:
        k.axis('off')
    ax = np.empty((nrows, 2), dtype=object)
    for r in range(nrows):
        for c in range(2):
            ax[r, c] = fig.add_subplot(gs[r + 2, c], sharex=ax[0, c] if (r and sharex) else None)
    return fig, keys, ax

def shared_legend(keys, axes, lines):
    """One legend per figure: the markers (by regime) on the first strip, the lines on the second."""
    marks = {}
    for a in axes:
        for h, label in zip(*a.get_legend_handles_labels()):
            marks.setdefault(label, h)
    keys[0].legend(list(marks.values()), list(marks), title='Simulation', loc='center', ncol=min(len(marks), 2))
    keys[1].legend(handles=lines, loc='center', ncol=len(lines))

def condition_lines(tags, fit_label):
    """Line key: one style per condition, or a single fit line when there is one condition."""
    if len(tags) == 1:
        return [Line2D([], [], color=INK, lw=2.4, label=fit_label)]
    return [Line2D([], [], color=INK, lw=2.4, ls=LINES[ci], label=f'{ROLES[ci]}: {tag}')
            for ci, tag in enumerate(tags)]

def end_label(ax, xv, y, tag, ci):
    """Name a curve at its right-hand end, staggered so coinciding curves stay readable."""
    ok = np.flatnonzero(np.isfinite(y))
    i = ok[np.argmax(xv[ok])]
    ax.annotate(tag, (xv[i], y[i]), xytext=(12, 14 * (ci - 1)), textcoords='offset points',
                va='center', fontsize=11, color=INK)

def fit_table(rows):
    """Every fitted line as one row: shown under the figure, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

def panel(ax, y, regime, by_regime, rows, quantity, A_units, tag, ls):
    """Simulation points plus Arrhenius fit(s), one group or one per regime; each fit becomes a table row."""
    for reg in (dict.fromkeys(regime) if by_regime else ['all']):
        m = (regime == reg) if by_regime else np.full(len(T), True)
        name, c, mk, *fill = look(reg) if by_regime else ('All temperatures (single fit)', GREY, 'o')
        hollow = fill == ['hollow']                      # Model 2: flux mainly through oxide defects
        ax.semilogy(x[m], y[m], mk, ms=9, color=c, mfc='white' if hollow else c,
                    mec=c if hollow else 'white', mew=2 if hollow else 1, label=name)
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        Ea, A, r2 = arrhenius(x[m], y[m])
        ax.semilogy(x[m], A * np.exp(-Ea / (R * T[m])), ls, lw=2.8, color=c if by_regime else INK)
        rows.append({'quantity': quantity, 'condition': tag, 'fit': name if by_regime else 'single fit',
                     'T window (K)': f'{T[m].min():.0f}–{T[m].max():.0f}', 'points': int(m.sum()),
                     'E_a (kJ/mol)': Ea / 1e3, 'A': A, 'A units': A_units, 'R²': r2})
    ax.grid(color=SHADE)

def arrhenius_figure(level, curves=None, note=None):
    """2x2 Arrhenius figure and its fit table. curves = [(tag, sweep output), ...], one per condition."""
    title, L = LEVELS[level]
    curves = curves or [(p_tag(P_up), sweep(level))]
    multi, units = len(curves) > 1, curves[0][1][2]
    quantities = [('J',       r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)',         'mol/m^2/s'),
                  ('Phi_app', rf'$\Phi_{{app}}$ ({UNITS_TEX[units]})',     units)]
    fig, keys, ax = framed(2, (14, 13))
    rows = []
    for r, (qty, ylab, A_units) in enumerate(quantities):
        for col in (0, 1):
            p = ax[r, col]
            for ci, (tag, out) in enumerate(curves):    # out = (J, Φ_app, units, regime)
                panel(p, out[r], out[3], col == 1, rows, qty, A_units, tag, LINES[ci] if multi else '-')
                if multi:
                    end_label(p, x, out[r], tag, ci)
            p.set_xlim(x.min() - 0.03, x.max() + (0.22 if multi else 0.03))   # room for the end labels
            p.set_ylabel(ylab)
    for p in ax[0]:
        p.tick_params(labelbottom=False)
        top = p.secondary_xaxis('top', functions=(lambda v: 1000/v - 273.15, lambda c: 1000/(c + 273.15)))
        top.set_xticks([300, 400, 500, 600, 700, 800, 1000])   # 900 °C would collide with 1000 °C
        top.set_xlabel('Temperature (°C)')
    for p in ax[1]:
        p.set_xlabel(r'1000/$T$ (K$^{-1}$)')
    ax[0, 0].set_title('Single Arrhenius fit (all temperatures)')
    ax[0, 1].set_title('Regime-resolved Arrhenius fits')
    shared_legend(keys, ax.flat, condition_lines([t for t, _ in curves],
                                                 'Arrhenius fit (black: single, colour: per regime)'))
    drive = r'\Delta P' if units == UNITS_HENRY else r'\Delta\sqrt{P}'
    where = note or rf'$P_{{up}}$ = {P_up:.0e} Pa'
    fig.suptitle(f'{title}\n{where},   ' + rf'$\Phi_{{app}} = J\,L/{drive}$,   $L$ = {L:.2e} m',
                 fontsize=18, fontweight='bold')
    plt.show()
    return fig, fit_table(rows)

##  4: Plot

In [ ]:
for level in LEVELS:
    fig, fits = arrhenius_figure(level)
    save_figure(fig, f"{level}_arrhenius", FIGS)
    fits.to_csv(FIGS / f"{level}_arrhenius_fits.csv", index=False)

In [ ]:
# =============================================================================
# MODEL 2 (L6 family): dissociation on the gas-facing surface, atomic H in oxide and metal
# Reuses solve() -> sweep() -> arrhenius_figure() above; only the solver is new.
# =============================================================================
from calculations.surface_kinetics import (get_all_properties, solve_steady_state_flux_L1L6,
                                           solve_steady_state_flux_L2aL6, solve_steady_state_flux_direct,
                                           calculate_mixed_defect_flux_L6, calculate_defective_metal_flux_L6,
                                           calculate_full_model_flux_L346_v2)
from calculations.config.model_config import METALS, get_surface_kinetics_from_config

MICRO   = SIM['microstructure']
DEFECTS = SIM['defect_config']
SIEVERTS_PINHOLE = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

LEVELS_M2 = {   # level: (title, wall thickness for Φ_app) -- every Model 2 level is Sieverts
    'L1L6':  ('Level 1+6: Perfect metal + surface kinetics',                     L_m),
    'L2aL6': ('Level 2a+6: Perfect oxide alone + surface kinetics',              L_ox),
    'L2bL6': ('Level 2b+6: Perfect oxide + perfect metal + surface kinetics',    L_ox + L_m),
    'L3L6':  ('Level 3+6: Defective oxide + perfect metal + surface kinetics',   L_ox + L_m),
    'L4L6':  ('Level 4+6: Perfect oxide + defective metal + surface kinetics',   L_ox + L_m),
    'L5L6':  ('Level 5+6: Defective oxide + defective metal + surface kinetics', L_ox + L_m),
}
LEVELS.update(LEVELS_M2)

def solve_m2(level, Tk):
    """Flux and regime label for one Model 2 level, e.g. 'oxide surface' or 'pinholes/metal diffusion'."""
    p  = get_all_properties(SIM['oxide_name'], SIM['metal_name'], Tk)       # oxide surface + transport
    sm = get_surface_kinetics_from_config(SIM['metal_name'], Tk, METALS)    # bare-metal surface
    k, Keq, Dox, Kox, Dm, Ksm = p['k_diss'], p['K_eq'], p['D_ox'], p['K_ox'], p['D_m'], p['K_s_m']
    pin  = {} if SIEVERTS_PINHOLE else dict(k_diss_metal=sm['k_diss'], K_eq_metal=sm['K_eq'])
    mode = MICRO.get('mode', 'both')
    if level == 'L1L6':  r = solve_steady_state_flux_L1L6(P_up, P_down, L_m, sm['k_diss'], sm['K_eq'], Dm, Ksm)
    if level == 'L2aL6': r = solve_steady_state_flux_L2aL6(P_up, P_down, k, Keq, Dox, Kox, L_ox)
    if level == 'L2bL6': r = solve_steady_state_flux_direct(P_up, P_down, L_m, k, Keq, Dox, Kox, L_ox, Dm, Ksm)
    if level == 'L3L6':  r = calculate_mixed_defect_flux_L6(P_up, P_down, L_m, k, Keq, Dox, Kox, L_ox,
                                                            Dm, Ksm, DEFECTS, **pin)
    if level == 'L4L6':  r = calculate_defective_metal_flux_L6(P_up, P_down, L_m, Tk, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, microstructure_params=MICRO, mode=mode)
    if level == 'L5L6':  r = calculate_full_model_flux_L346_v2(P_up, P_down, L_m, Tk, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, MICRO, DEFECTS, mode=mode, **pin)
    return r.get('J_total', r.get('J_ss', r.get('flux'))), label_model2(level, r, DEFECTS)

for level in LEVELS_M2:
    fig, fits = arrhenius_figure(level)
    save_figure(fig, f"{level}_arrhenius", FIGS)
    fits.to_csv(FIGS / f"{level}_arrhenius_fits.csv", index=False)

## Three pressures: the temperature sweep at the low / mid / high of `P_range`

In [ ]:
# =============================================================================
# THREE PRESSURES: the temperature sweep at the low / mid / high of P_range
# Same figure as above, with one curve per pressure in each panel. Colour and marker still mean
# the regime; line style and the label at the end of each curve mean the pressure.
# =============================================================================
from contextlib import contextmanager

P_lo, P_hi = CONDITIONS['P_range']
P_SET = [P_lo, np.sqrt(P_lo * P_hi), P_hi]          # low / mid (log scale) / high: 1e-7, 1, 1e7 Pa

@contextmanager
def at_pressure(Pc):
    """Run solve() and sweep() above at another upstream pressure; P_up is restored afterwards."""
    global P_up
    saved, P_up = P_up, Pc
    try:
        yield
    finally:
        P_up = saved

for level in LEVELS:
    curves = []
    for Pc in P_SET:
        with at_pressure(Pc):
            curves.append((p_tag(Pc), sweep(level)))
    fig, fits = arrhenius_figure(level, curves, note='three pressures (low / mid / high of P_range)')
    save_figure(fig, f"{level}_arrhenius_3P", FIGS)
    fits.to_csv(FIGS / f"{level}_arrhenius_3P_fits.csv", index=False)